# Análise Institucional de Municípios — Cidades do Futuro

Notebook enxuto: instala dependências, monta o Drive e executa o pacote modular `municipio_analise` (classes, agente LangChain com memória, chamadas assíncronas, etc.). Toda a lógica está no pacote, não neste notebook.


In [ ]:
!pip install -q -r /content/drive/MyDrive/Scripts/municipio_analise/requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Configuração

Ajuste `PACKAGE_PARENT_PATH` para a pasta do Drive onde você colocou a pasta `municipio_analise/` (o pacote), e `BASE_PATH` para a pasta onde está `indicadores.xlsx` e onde os relatórios `.docx` serão salvos.


Os índices FAISS de tópicos e chunks são criados na primeira execução e persistidos em `BASE_PATH` para reutilização nas próximas execuções.


In [ ]:
import sys
import os

PACKAGE_PARENT_PATH = "/content/drive/MyDrive/Scripts"  # pasta que CONTÉM municipio_analise/
BASE_PATH = "/content/drive/MyDrive/Scripts/Analise de municipio"

if PACKAGE_PARENT_PATH not in sys.path:
    sys.path.insert(0, PACKAGE_PARENT_PATH)

# Recomendado: use Colab Secrets (ícone de chave 🔑 na barra lateral) em vez
# de deixar a chave em texto puro.
from google.colab import userdata
try:
    # Compatível com o notebook de referência, que usa o secret chamado API.
    os.environ["GOOGLE_API_KEY"] = userdata.get("API")
except Exception:
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        print("⚠️  Defina API (ou GOOGLE_API_KEY) em Colab Secrets.")


## Execução direta (sem agente) — gera o relatório de um município

In [ ]:
import asyncio
from municipio_analise import Config, PipelineAnaliseMunicipio

config = Config(base_path=BASE_PATH)
pipeline = PipelineAnaliseMunicipio(config)
pipeline.carregar_planilha()

print("Municípios disponíveis:")
for i, m in enumerate(pipeline.leitor.listar_municipios(), 1):
    print(f"{i} - {m}")


In [ ]:
MUNICIPIO_ESCOLHIDO = ""  # preencha com o nome exato de um dos municípios acima

relatorio = await pipeline.analisar_municipio(MUNICIPIO_ESCOLHIDO)
caminho = pipeline.salvar_relatorio(relatorio)
print(f"📄 Relatório salvo em: {caminho}")


## Alternativa: agente conversacional com memória

Permite perguntar em linguagem natural ("quais municípios existem?", "gere o relatório de X", "e a dimensão sociocultural desse último município?") mantendo o contexto da conversa.


In [ ]:
from municipio_analise import AgenteAnaliseMunicipal

agente = AgenteAnaliseMunicipal(pipeline)

resposta = await agente.perguntar("Quais municípios estão disponíveis?")
print(resposta)


In [ ]:
resposta = await agente.perguntar("Gere o relatório institucional do primeiro município da lista.")
print(resposta)
